# 🎬 YOHA VIDEO · YDC Entertainment Virales — Estudio de generación gratuito (v2)

Convierte la producción creada en la app **Yoha Video** (o una sola instrucción) en clips de video con IA, voces, subtítulos y, opcionalmente, labios sincronizados. Todo con **modelos abiertos** y la **GPU gratuita** de Kaggle o Google Colab. **Costo: US$0** (sin tarjeta, sin APIs de pago).

### Pasos (sin experiencia técnica)
1. **Kaggle** (recomendado, ~30 h de GPU por semana con teléfono verificado): kaggle.com → *Create → New Notebook* → *File → Import Notebook* → este archivo. Panel derecho: **Accelerator → GPU T4 x2**, **Internet → On**.
   **Colab**: colab.research.google.com → *Subir* → *Entorno de ejecución → Cambiar tipo → T4 GPU* (cuota variable).
2. Ejecuta las celdas en orden con ▶. En la **celda 3** pega tu instrucción o sube `yoha-produccion-….json` (pestaña *Crear video → producción → Paquete*).
3. Descarga **`yoha_clips.zip`** al final y ábrelo en Yoha Video → *Editor*. Agrega intro/cierre YDC y música, exporta y envía a revisión.

### Motores (verificados como de uso comercial permitido; revisa sus licencias antes de publicar)
| Etapa | Motor | Licencia | Estado en GPU gratuita |
|---|---|---|---|
| Guion y diálogos | **Qwen2.5-7B-Instruct** (4 bits) | Apache-2.0 | ✔ cabe en T4 |
| Video | **Wan 2.1 T2V 1.3B** (480p, ~5 s) | Apache-2.0 | ✔ T4 |
| Video (mejor calidad) | **Wan 2.2 TI2V 5B** (720p) | Apache-2.0 | ⚠ experimental y lento en T4 |
| Voces ES / EN | **Kokoro 82M** | Apache-2.0 | ✔ CPU/GPU |
| Subtítulos desde el audio | **faster-whisper** (Whisper small) | MIT | ✔ |
| Labios sincronizados | **LatentSync 1.5** | OpenRAIL++ (uso comercial con restricciones de uso) | ⚠ experimental, apagado por defecto |
| Continuidad de personajes | **CLIP ViT-B/32** (parecido con la foto) | MIT | ✔ |
| Escalado 1080p/4K | **FFmpeg lanczos** | LGPL | ✔ (es **escalado**, no 4K nativo) |

Descartados: LivePortrait (sus modelos de detección facial de InsightFace no permiten uso comercial), LTX-Video (licencia no verificada para tu caso), MusicGen (no comercial).

> ⚠️ Realidad: cada clip dura ~5 s y tarda varios minutos en T4. Una novela de 20 min son ~240 clips → varias sesiones de cuota gratuita. El cuaderno **reanuda** donde quedó y **solo regenera las escenas modificadas**. La actuación (emociones, gestos, caminar) se dirige con el texto; el control es aproximado y la consistencia de rostros no es perfecta: el cuaderno la mide y marca escenas a revisar.

## 1 · GPU disponible

In [ ]:
import subprocess, shutil, sys, os, time
T_INICIO = globals().get("T_INICIO") or time.time()
print("Python", sys.version.split()[0])
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else "❌ Sin GPU: actívala (Kaggle: Accelerator → GPU T4 x2 · Colab: Cambiar tipo → T4 GPU).")
try:
    import torch
    GPU_OK = torch.cuda.is_available()
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if GPU_OK else 0
    print(f"{'✅' if GPU_OK else '❌'} PyTorch CUDA: {GPU_OK} · VRAM {VRAM_GB:.1f} GB")
except ImportError:
    GPU_OK, VRAM_GB = False, 0

## 2 · Instalar dependencias (3–6 min la primera vez)

In [ ]:
import subprocess, sys
def sh(cmd, check=True):
    print("$", cmd[:120]); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode and check: print(r.stdout[-1500:], r.stderr[-1500:]); raise SystemExit(f"Falló: {cmd}")
    return r
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y espeak-ng ffmpeg >/dev/null 2>&1", check=False)
sh(f"{sys.executable} -m pip install -q -U 'diffusers>=0.35' 'transformers>=4.49' accelerate bitsandbytes ftfy imageio imageio-ffmpeg sentencepiece soundfile open_clip_torch faster-whisper requests uv")
# Voces Kokoro: requieren Python < 3.13. Se instalan en un entorno propio con Python 3.12 (uv lo descarga) y torch CPU.
TTS_PY = "/tmp/tts/bin/python"
r = sh("uv venv -q -p 3.12 /tmp/tts && uv pip install -q -p /tmp/tts/bin/python 'kokoro>=0.9.4' soundfile 'pip' "
       "torch --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match", check=False)
print("✅ entorno de voces (Python 3.12)" if r.returncode == 0 else "⚠️ Voces no disponibles: " + r.stderr[-400:])
try:
    from diffusers import WanPipeline  # noqa: F401
except Exception:
    sh(f"{sys.executable} -m pip install -q -U git+https://github.com/huggingface/diffusers")
import diffusers, torch
print("✅ diffusers", diffusers.__version__, "· torch", torch.__version__)

## 3 · Producción
- **MODO = "paquete"** (recomendado): sube el `yoha-produccion-….json` exportado desde Yoha Video. Respeta reparto, escenas, planos, actuación, voces, idiomas y solo regenera lo modificado.
- **MODO = "instruccion"**: escribe una sola instrucción y el guionista IA arma todo aquí.
- **AVISO_NTFY**: tema de la app gratuita **ntfy** (Yoha Video → Configuración) para recibir en el iPhone «🎬 Yoha Video: tu producción está lista para revisar».

In [ ]:
MODO = "paquete"            #@param ["paquete", "instruccion"]
INSTRUCCION = "Créame una novelita dominicana de dos minutos, con tres personajes, romance, traición y un final que motive a ver el próximo capítulo"  #@param {type:"string"}
ESCRIBIR_GUION_CON_IA = True #@param {type:"boolean"}
MODELO_VIDEO = "auto"       #@param ["auto", "wan2.1-1.3b", "wan2.2-5b"]
CALIDAD = "normal"          #@param ["borrador", "normal", "alta"]
MAX_PLANOS_ESTA_SESION = 24 #@param {type:"integer"}
VOCES = True                #@param {type:"boolean"}
SUBTITULOS_WHISPER = True   #@param {type:"boolean"}
LABIOS_SINCRONIZADOS = False #@param {type:"boolean"}
REVISAR_PARECIDO = True     #@param {type:"boolean"}
UMBRAL_PARECIDO = 0.72      #@param {type:"number"}
ESCALAR_A = "ninguno"       #@param ["ninguno", "1080p", "4k"]
AVISO_NTFY = ""             #@param {type:"string"}
LIMITE_HORAS = 11.0         #@param {type:"number"}

# Modo automático: la Productora de Yoha Video (GitHub Actions) antepone una celda con YOHA_AUTO = {...}
YOHA_AUTO = globals().get("YOHA_AUTO")
if YOHA_AUTO:
    MODO = "paquete"
    globals().update(YOHA_AUTO.get("params", {}))

import json, os, glob, pathlib, re, requests, hashlib
OUT = pathlib.Path("/kaggle/working/yoha_clips" if os.path.isdir("/kaggle/working") else "/content/yoha_clips"); OUT.mkdir(parents=True, exist_ok=True)

def avisar(kind, detalle=""):
    msgs = {"ready": "🎬 Yoha Video: tu producción está lista para revisar.", "error": "⚠️ Yoha Video: ocurrió un error crítico en tu producción.", "stopped": "⏸️ Yoha Video: la producción se detuvo antes de terminar."}
    if not re.fullmatch(r"[A-Za-z0-9_-]{12,64}", AVISO_NTFY or ""): return
    try: requests.post(f"https://ntfy.sh/{AVISO_NTFY}", data=(msgs[kind] + ("\n" + detalle if detalle else "")).encode(), headers={"Title": "Yoha Video", "Priority": "default" if kind == "ready" else "high"}, timeout=10)
    except Exception as e: print("Aviso no enviado:", e)

STYLE = "cinematic film still, natural skin texture, soft key light, shallow depth of field, 35mm, color graded, telenovela production quality"
EMO = {"neutral": "calm neutral expression", "feliz": "smiling warmly, happy", "risa": "laughing naturally", "triste": "sad, glossy eyes", "llanto": "crying, tears running down the cheeks", "enfado": "angry, furrowed brow, clenched jaw", "sorpresa": "surprised, eyes wide open", "miedo": "afraid, tense", "amor": "loving gaze, soft smile", "celos": "jealous, narrowed eyes", "determinacion": "determined, steady gaze"}
GEST = {"ninguno": "", "asiente": "nods slowly", "niega": "shakes head", "manos": "gestures with both hands while talking", "abrazo": "hugs the other person", "señala": "points at someone", "cubre_rostro": "covers face with hands", "toma_mano": "holds the other person's hand", "brazos_cruzados": "crosses arms"}
MOVE = {"quieto": "", "camina": "walks slowly toward the camera", "entra": "enters the room through the door", "se_va": "turns and walks away", "se_sienta": "sits down", "se_levanta": "stands up abruptly", "gira": "turns to face the other character", "corre": "runs"}

if YOHA_AUTO:
    PKG = YOHA_AUTO["pkg"]
    assert PKG.get("format") == "yoha-production", "No es un paquete de Yoha Video."
    # Reanudar: clips y guion de la sesión anterior (salida del kernel previo, añadida como entrada)
    import shutil as _sh
    prev = [pathlib.Path(f) for f in glob.glob("/kaggle/input/**/yoha_clips/*", recursive=True)]
    for f in prev:
        if f.suffix == ".mp4" and not (OUT / f.name).exists(): _sh.copy(f, OUT / f.name)
    g = next((f for f in prev if f.name == "guion_ia.json"), None)
    if g and not any(x.get("dialogue") for x in PKG["scenes"]):
        G = json.load(open(g, encoding="utf-8"))
        PKG["title"], PKG["synopsis"] = G.get("title", PKG.get("title")), G.get("synopsis", PKG.get("synopsis", ""))
        if G.get("characters"): PKG["characters"] = G["characters"]
        PKG["scenes"] = G["scenes"]; _sh.copy(g, OUT / "guion_ia.json")
    print(f"Modo automático · {len([f for f in prev if f.suffix == '.mp4'])} clips de la sesión anterior")
elif MODO == "paquete":
    files = glob.glob("/kaggle/input/**/yoha-*.json", recursive=True)
    if not files:
        try:
            from google.colab import files as gf  # type: ignore
            print("Sube tu yoha-produccion-….json"); files = list(gf.upload().keys())
        except ImportError:
            raise SystemExit("Kaggle: Add Input → Upload → sube el JSON de Yoha Video y vuelve a ejecutar.")
    PKG = json.load(open(files[0], encoding="utf-8"))
    assert PKG.get("format") == "yoha-production", "No es un paquete de Yoha Video."
else:
    PKG = {"format": "yoha-production", "version": 2, "title": "Producción YDC", "instruction": INSTRUCCION, "styleText": STYLE, "aspect": "9:16" if re.search(r"short|vertical|tiktok|reel", INSTRUCCION, re.I) else "16:9", "languages": ["es"] + (["en"] if re.search(r"ingl|english|doblaj", INSTRUCCION, re.I) else []), "allowStrongLanguage": False, "characters": [], "scenes": [], "brief": None}
STYLE = PKG.get("styleText") or STYLE
print(f"«{PKG.get('title')}» · {len(PKG['scenes'])} escenas · {len(PKG['characters'])} personajes · idiomas {PKG.get('languages')} · {'solo escenas modificadas' if PKG.get('onlyChanged') else 'completa'}")

## 4 · Director IA: guion completo con Qwen2.5-7B (gratis, lenguaje limpio por defecto)

In [ ]:
BAD = ["mierda","carajo","coño","joder","puta","puto","pendejo","pendeja","cabron","cabrón","chingar","verga","huevon","maricon","gilipollas","culero","mamaguevo","hijo de puta","comemierda","singar","fuck","shit","bitch","bastard","asshole","motherfucker","cunt","whore","slut"]
def groserias(t):
    t = (t or "").lower(); return [w for w in BAD if re.search(rf"(^|\W){re.escape(w)}(?=$|\W)", t)]

def build_brief():
    if PKG.get("brief"): return PKG["brief"]
    return f"""Eres guionista profesional para el canal YDC Entertainment Virales. Instrucción: "{PKG['instruction']}".
Crea una historia 100% original en español, lenguaje limpio (sin malas palabras). Define 2 a 6 personajes con id cortos (p1, p2…), escenas de 15-40 s divididas en planos de ~5 s, diálogos breves.
Responde SOLO JSON: {{"title":"","synopsis":"","characters":[{{"id":"p1","name":"","role":"","appearance":"descripción física en inglés","wardrobe":"en inglés","voice":"ef_dora|em_alex|em_santa","voiceEn":"af_heart|am_michael"}}],"scenes":[{{"id":"e1","title":"","location":"","time":"día|atardecer|noche","description":"","descriptionEn":"visual description in English","camera":"","characters":["p1"],"shots":3,"narration":"","dialogue":[{{"character":"p1","text":"","emotion":"neutral|feliz|risa|triste|llanto|enfado|sorpresa|miedo|amor|celos|determinacion"}}],"dialogueEn":[{{"character":"p1","text":""}}],"musicMood":"","sfx":""}}]}}"""

def write_script():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    mid = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(mid)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    llm = AutoModelForCausalLM.from_pretrained(mid, quantization_config=q, device_map="auto")
    brief = build_brief()
    for intento in range(2):
        msgs = [{"role": "system", "content": "Respondes solo con JSON válido. Historias originales, sin lenguaje vulgar salvo autorización."}, {"role": "user", "content": brief}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
        out = llm.generate(ids, max_new_tokens=6000, temperature=0.8, top_p=0.9, do_sample=True)
        text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        m = re.search(r"\{[\s\S]*\}", text)
        try:
            data = json.loads(m.group(0))
        except Exception:
            print("JSON inválido, reintentando…"); continue
        malas = groserias(json.dumps(data, ensure_ascii=False))
        if malas and not PKG.get("allowStrongLanguage"):
            print("Lenguaje no permitido:", malas, "→ reescribiendo"); brief += "\nIMPORTANTE: elimina cualquier mala palabra."; continue
        break
    del llm; torch.cuda.empty_cache()
    return data

if ESCRIBIR_GUION_CON_IA and (MODO == "instruccion" or not any(s.get("dialogue") for s in PKG["scenes"])):
    if not torch.cuda.is_available(): raise SystemExit("🚫 BLOQUEADO: el guionista necesita GPU.")
    data = write_script()
    if MODO == "instruccion":
        PKG["title"], PKG["synopsis"] = data.get("title", PKG["title"]), data.get("synopsis", "")
        PKG["characters"] = [{**c, "seed": 1000 + i * 101, "references": []} for i, c in enumerate(data.get("characters", []))]
        PKG["scenes"] = [{**s, "n": i + 1, "seed": 2000 + i * 37, "seconds": int(s.get("shots", 3)) * 5, "acting": {}} for i, s in enumerate(data.get("scenes", []))]
    else:
        by_id = {s["id"]: s for s in data.get("scenes", [])}
        for s in PKG["scenes"]:
            n = by_id.get(s["id"])
            if n: s.update({k: n[k] for k in ("title", "description", "descriptionEn", "narration", "dialogue", "dialogueEn", "musicMood", "sfx") if k in n})
    json.dump({"title": PKG["title"], "synopsis": PKG.get("synopsis", ""), "characters": PKG["characters"] if MODO == "instruccion" else None, "scenes": PKG["scenes"]}, open(OUT / "guion_ia.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("✅ Guion escrito. guion_ia.json se incluye en el .zip (impórtalo en Yoha Video → producción → «Importar guion»).")
else:
    print("Se usa el guion del paquete.")

# Dirección automática: texto final de cada plano con apariencia fija + actuación + cámara
CH = {c["id"]: c for c in PKG["characters"]}
def prompt_for(s, k):
    people = []
    for cid in s.get("characters", []):
        c = CH.get(cid);
        if not c: continue
        a = (s.get("acting") or {}).get(cid, {})
        emo = a.get("emotion") or next((d.get("emotion") for d in s.get("dialogue", []) if d.get("character") == cid and d.get("emotion")), "neutral")
        cue = ", ".join(x for x in [EMO.get(emo, ""), GEST.get(a.get("gesture", ""), ""), MOVE.get(a.get("move", ""), "")] if x)
        people.append(f"{c['name']}, {c['appearance']}" + (f", wearing {c['wardrobe']}" if c.get("wardrobe") else "") + (f" — {cue}" if cue else ""))
    light = {"noche": "night", "atardecer": "golden hour sunset light"}.get(s.get("time"), "daylight")
    talk = " The characters talk to each other, lips moving naturally." if s.get("dialogue") else ""
    return ". ".join(x for x in [f"{s.get('location', '')}, {light}", s.get("descriptionEn") or s.get("description", ""), people and "Characters: " + "; ".join(people), f"Camera: {s.get('camera', 'medium shot')}" + (", reverse angle" if k % 2 else ""), STYLE] if x) + "." + talk
for s in PKG["scenes"]:
    s["hash"] = s.get("hash") or hashlib.sha1(json.dumps([s.get("descriptionEn") or s.get("description"), s.get("characters"), s.get("acting"), s.get("seed"), s.get("shots")], sort_keys=True).encode()).hexdigest()[:8]
print("Ejemplo de dirección:\n", prompt_for(PKG["scenes"][0], 0)[:600])

## 5 · Modelo de video (elige solo según tu GPU)

In [ ]:
import gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
if not torch.cuda.is_available():
    avisar("stopped", "No hay GPU disponible."); raise SystemExit("🚫 BLOQUEADO: no hay GPU. No se generan videos simulados.")
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
cap = torch.cuda.get_device_capability(0)
choice = MODELO_VIDEO if MODELO_VIDEO != "auto" else ("wan2.2-5b" if vram >= 22 else "wan2.1-1.3b")
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
SPECS = {"wan2.1-1.3b": ("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", 16, 81, (832, 480)), "wan2.2-5b": ("Wan-AI/Wan2.2-TI2V-5B-Diffusers", 24, 121, (1280, 704))}
REPO, FPS, FRAMES, (W0, H0) = SPECS[choice]
if choice == "wan2.2-5b" and vram < 22: W0, H0, FRAMES = 832, 480, 81
W, H = {"16:9": (W0, H0), "9:16": (H0, W0), "1:1": (H0, H0), "4:5": (H0, int(H0 * 1.25) // 32 * 32)}.get(PKG.get("aspect", "16:9"), (W0, H0))
STEPS = {"borrador": 20, "normal": 30, "alta": 50}[CALIDAD]
vae = AutoencoderKLWan.from_pretrained(REPO, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(REPO, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload() if vram < 22 else pipe.to("cuda")
try: pipe.vae.enable_tiling()
except Exception: pass
NEG = "blurry, low quality, distorted face, deformed hands, extra fingers, extra limbs, text, watermark, logo, static frame, flicker, bad anatomy"
print(f"✅ {choice} · {W}x{H} · {FRAMES} cuadros @ {FPS} fps · {STEPS} pasos · {DTYPE}")

## 6 · Generar planos (reanuda; solo regenera escenas modificadas; mide parecido con la foto del personaje)

In [ ]:
import time, numpy as np, base64, io
from PIL import Image
refs = {c["id"]: [Image.open(io.BytesIO(base64.b64decode(d.split(",", 1)[1]))).convert("RGB") for d in c.get("references", [])] for c in PKG["characters"] if (c.get("consent") or {}).get("given")}
clip_m = None
def parecido(frames, ref_imgs):
    global clip_m
    import open_clip
    if clip_m is None:
        m, _, pre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai"); clip_m = (m.eval().to("cuda"), pre)
    m, pre = clip_m
    to_img = lambda f: Image.fromarray((np.clip(np.asarray(f), 0, 1) * 255).astype("uint8") if np.asarray(f).dtype != np.uint8 else np.asarray(f))
    with torch.no_grad():
        a = m.encode_image(torch.stack([pre(to_img(frames[i])) for i in (len(frames)//4, len(frames)//2, -1)]).to("cuda"))
        b = m.encode_image(torch.stack([pre(r) for r in ref_imgs]).to("cuda"))
    a = a / a.norm(dim=-1, keepdim=True); b = b / b.norm(dim=-1, keepdim=True)
    return float((a @ b.T).mean())

report, hechos = [], 0
try:
    for s in PKG["scenes"]:
        for k in range(int(s.get("shots", 1))):
            out = OUT / f"e{int(s['n']):03d}_p{k+1:02d}_{s['hash']}.mp4"
            if out.exists() and out.stat().st_size > 10_000:
                report.append({"scene": s["id"], "shot": k + 1, "file": out.name, "status": "existente"}); continue
            if time.time() - T_INICIO > LIMITE_HORAS * 3600:
                print(f"⏸  Se alcanzó el límite de {LIMITE_HORAS} h de esta sesión de GPU. Se continúa en la siguiente."); raise StopIteration
            if hechos >= MAX_PLANOS_ESTA_SESION:
                print(f"⏸  Límite de {MAX_PLANOS_ESTA_SESION} planos en esta sesión. Vuelve a ejecutar para continuar."); raise StopIteration
            cast_refs = [im for cid in s.get("characters", []) for im in refs.get(cid, [])]
            best = None
            for intento in range(2 if (REVISAR_PARECIDO and cast_refs) else 1):
                seed = int(s.get("seed", 1)) + k * 101 + intento * 7919
                t0 = time.time()
                frames = pipe(prompt=prompt_for(s, k), negative_prompt=NEG, height=H, width=W, num_frames=FRAMES, num_inference_steps=STEPS, guidance_scale=5.0, generator=torch.Generator("cuda").manual_seed(seed)).frames[0]
                score = parecido(frames, cast_refs) if (REVISAR_PARECIDO and cast_refs) else None
                print(f"🎞  Escena {s['n']} plano {k+1}: {time.time()-t0:.0f}s · semilla {seed}" + (f" · parecido {score:.2f}" if score is not None else ""))
                if best is None or (score is not None and score > best[1]): best = (frames, score, seed)
                if score is None or score >= UMBRAL_PARECIDO: break
            frames, score, seed = best
            export_to_video(frames, str(out), fps=FPS); hechos += 1
            report.append({"scene": s["id"], "shot": k + 1, "file": out.name, "seed": seed, "similarity": score, "needs_review": bool(score is not None and score < UMBRAL_PARECIDO)})
            gc.collect(); torch.cuda.empty_cache()
except StopIteration:
    if not YOHA_AUTO: avisar("stopped", f"Se generaron {hechos} planos; faltan más. Vuelve a ejecutar la celda 6.")
except Exception as e:
    avisar("error", str(e)[:200]); raise
json.dump(report, open(OUT / "report.json", "w"), indent=1, ensure_ascii=False)
print("✅ Planos nuevos en esta sesión:", hechos)

## 7 · Voces (Kokoro) en español e inglés, alineadas a cada escena

In [ ]:
import soundfile as sf
SR = 24000
def dur(p): r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(p)],capture_output=True,text=True); return float(r.stdout.strip() or 0)
def shots_of(s): return sorted(OUT.glob(f"e{int(s['n']):03d}_p*_{s['hash']}.mp4"))
def voice_track(lang):
    import soundfile as _sf
    key = "dialogue" if lang == "es" else "dialogueEn"
    pend = []  # (texto, voz) de todas las escenas → una sola llamada al entorno de voces
    for s in PKG["scenes"]:
        nar = s.get("narration") if lang == "es" else s.get("narrationEn")
        for cid, text in ([(None, nar)] if nar else []) + [(x.get("character"), x.get("text")) for x in s.get(key, []) if x.get("text")]:
            c = CH.get(cid, {}); pend.append({"text": text, "voice": (c.get("voice") or "ef_dora") if lang == "es" else (c.get("voiceEn") or "af_heart")})
    if not pend: return []
    tdir = OUT.parent / f"_tts_{lang}"; tdir.mkdir(exist_ok=True)
    json.dump(pend, open(tdir / "lineas.json", "w", encoding="utf-8"), ensure_ascii=False)
    (tdir / "hablar.py").write_text("""
import json, sys, numpy as np, soundfile as sf
from kokoro import KPipeline
d = sys.argv[1]; tts = KPipeline(lang_code=sys.argv[2], repo_id="hexgrad/Kokoro-82M")
for i, x in enumerate(json.load(open(d + "/lineas.json", encoding="utf-8"))):
    try: a = np.concatenate([np.asarray(t, np.float32) for _, _, t in tts(x["text"], voice=x["voice"])])
    except Exception as e:
        print("voz alternativa para", x["voice"], e); a = np.concatenate([np.asarray(t, np.float32) for _, _, t in tts(x["text"], voice="ef_dora" if sys.argv[2] == "e" else "af_heart")])
    sf.write(f"{d}/l{i:04d}.wav", a, 24000)
print("ok", i + 1)
""", encoding="utf-8")
    r = sh(f"{TTS_PY} {tdir / 'hablar.py'} {tdir} {'e' if lang == 'es' else 'a'}", check=False)
    if r.returncode: raise RuntimeError("Voces: " + (r.stderr or r.stdout)[-600:])
    audios = iter([_sf.read(tdir / f"l{i:04d}.wav", dtype="float32")[0] for i in range(len(pend))])
    track, t, subs = np.zeros(0, np.float32), 0.0, []
    for s in PKG["scenes"]:
        d = sum(dur(p) for p in shots_of(s)) or s.get("seconds", 5)
        lines = ([(None, s.get("narration") if lang == "es" else s.get("narrationEn", ""))] if (s.get("narration") if lang == "es" else s.get("narrationEn")) else []) + [(x.get("character"), x.get("text")) for x in s.get(key, []) if x.get("text")]
        pos = t + 0.3
        for cid, text in lines:
            c = CH.get(cid, {}); audio = next(audios)
            a0 = int(pos * SR); a1 = a0 + len(audio)
            if len(track) < a1: track = np.pad(track, (0, a1 - len(track)))
            track[a0:a1] += audio
            subs.append((pos, pos + len(audio) / SR, (c.get("name", "") + ": " if c else "") + text)); pos += len(audio) / SR + 0.25
        t += d
    if track.size: sf.write(OUT / f"voces_{lang}.wav", np.clip(track, -1, 1), SR)
    srt = "\n".join(f"{i+1}\n{fmt(a)} --> {fmt(b)}\n{x}\n" for i, (a, b, x) in enumerate(subs))
    (OUT / f"subtitulos_{lang}.srt").write_text(srt, encoding="utf-8")
    print(f"✅ voces_{lang}.wav · {track.size/SR:.1f}s de voz · video {t:.1f}s"); return subs
def fmt(x): ms = int(x * 1000); return f"{ms//3600000:02d}:{ms//60000%60:02d}:{ms//1000%60:02d},{ms%1000:03d}"
SUBS = {}
if VOCES:
    for lang in PKG.get("languages", ["es"]):
        if lang == "en" and not any(s.get("dialogueEn") or s.get("narrationEn") for s in PKG["scenes"]):
            print("Inglés: el guion no trae diálogos en inglés (pídelos al guionista IA)."); continue
        try:
            SUBS[lang] = voice_track(lang)
        except Exception as e:  # sin voces la producción sigue: el video se monta con silencio y se avisa
            print(f"⚠️ No se generaron las voces ({lang}): {e}")

## 8 · (Opcional) Subtítulos desde el audio con Whisper · (Experimental) labios sincronizados con LatentSync

In [ ]:
def _whisper():
    from faster_whisper import WhisperModel
    wm = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
    segs, _ = wm.transcribe(str(OUT / "voces_es.wav"), language="es")
    (OUT / "subtitulos_whisper_es.srt").write_text("\n".join(f"{i+1}\n{fmt(s.start)} --> {fmt(s.end)}\n{s.text.strip()}\n" for i, s in enumerate(segs)), encoding="utf-8")
    print("✅ subtitulos_whisper_es.srt (verificación de lo que realmente se oye)")
if SUBTITULOS_WHISPER and (OUT / "voces_es.wav").exists():
    try: _whisper()
    except Exception as e: print("⚠️ Whisper no disponible:", e)

if LABIOS_SINCRONIZADOS:
    # EXPERIMENTAL: LatentSync 1.5 (OpenRAIL++). Requiere GPU; en T4 puede ser lento o quedarse sin memoria.
    if not pathlib.Path("LatentSync").exists():
        sh("git clone -q --depth 1 https://github.com/bytedance/LatentSync.git")
        sh(f"cd LatentSync && {sys.executable} -m pip install -q -r requirements.txt", check=False)
        sh(f"{sys.executable} -c \"from huggingface_hub import snapshot_download; snapshot_download('ByteDance/LatentSync-1.5', local_dir='LatentSync/checkpoints')\"")
    wav = OUT / "voces_es.wav"
    if wav.exists():
        t = 0.0
        for s in PKG["scenes"]:
            for p in shots_of(s):
                d = dur(p)
                if s.get("dialogue"):
                    seg = OUT / f"_a_{p.stem}.wav"
                    sh(f"ffmpeg -v error -y -ss {t} -t {d} -i {wav} -ar 16000 -ac 1 {seg}")
                    out = p.with_name(p.stem + "_lips.mp4")
                    r = sh(f"cd LatentSync && {sys.executable} -m scripts.inference --unet_config_path configs/unet/stage2.yaml --inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps 20 --guidance_scale 1.5 --video_path {p} --audio_path {seg} --video_out_path {out}", check=False)
                    if out.exists(): out.replace(p); print("👄", p.name)
                    else: print("⚠️ LatentSync falló en", p.name, r.stderr[-300:])
                t += d

## 9 · Escalado opcional (escalado, no 4K nativo) · empaquetar · avisar al iPhone

In [ ]:
import zipfile
from IPython.display import Video, display
clips = [p for s in PKG["scenes"] for p in shots_of(s)]
if ESCALAR_A != "ninguno":
    short = 1080 if ESCALAR_A == "1080p" else 2160
    for p in clips:
        tmp = p.with_name(p.stem + ".up.mp4")
        sh(f"ffmpeg -v error -y -i {p} -vf scale={'-2:'+str(short) if W >= H else str(short)+':-2'}:flags=lanczos,unsharp=5:5:0.4 -c:v libx264 -crf 18 -pix_fmt yuv420p {tmp}")
        tmp.replace(p)
for p in clips[:3]: display(Video(str(p), embed=True, width=420))
def sub_for(s, k):
    lines = [d.get("text", "") for d in s.get("dialogue", [])]
    if not lines: return s.get("narration", "") if k == 0 else ""
    per = max(1, -(-len(lines) // int(s.get("shots", 1))))
    return " ".join(lines[k * per:(k + 1) * per])
manifest = {"format": "yoha-clips", "version": 2, "generator": f"{choice} · {W}x{H}@{FPS}fps · {STEPS} pasos", "upscaled": ESCALAR_A, "production": PKG.get("id"), "title": PKG.get("title"),
            "clips": [], "narration": "voces_es.wav" if (OUT / "voces_es.wav").exists() else None, "narrationEn": "voces_en.wav" if (OUT / "voces_en.wav").exists() else None}
for s in PKG["scenes"]:
    for k, p in enumerate(shots_of(s)):
        manifest["clips"].append({"file": p.name, "scene": s["id"], "title": PKG.get("title", "") if (s["n"] == 1 and k == 0) else "", "subtitle": sub_for(s, k), "seconds": round(dur(p), 2)})
json.dump(manifest, open(OUT / "manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
total = len([p for s in PKG["scenes"] for p in range(int(s.get('shots', 1)))])
json.dump({"completo": len(clips) >= total, "planos_hechos": len(clips), "planos_total": total, "titulo": PKG.get("title"), "w": W, "h": H, "fps": FPS, "modelo": choice}, open(OUT / "estado_sesion.json", "w", encoding="utf-8"), ensure_ascii=False)
if YOHA_AUTO:
    print(f"✅ Sesión terminada · {len(clips)}/{total} planos · la Productora de Yoha Video monta el video final.")
else:
    zpath = OUT.parent / "yoha_clips.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f in [*clips, *OUT.glob("*.wav"), *OUT.glob("*.srt"), OUT / "manifest.json", OUT / "report.json", OUT / "guion_ia.json"]:
            if f.exists() and not f.name.startswith("_a_"): z.write(f, f"yoha_clips/{f.name}")
    print(f"✅ {zpath} · {zpath.stat().st_size/1e6:.1f} MB · {len(clips)}/{total} planos")
    if len(clips) >= total: avisar("ready", f"{PKG.get('title')}: {len(clips)} planos listos.")
    else: avisar("stopped", f"{len(clips)}/{total} planos. Ejecuta de nuevo la celda 6 en otra sesión.")
    try:
        from google.colab import files as gf  # type: ignore
        gf.download(str(zpath))
    except ImportError:
        print("Kaggle: panel Output → yoha_clips.zip → Download. Ábrelo en Yoha Video → Editor.")